### Set Notebook Constants

In [ ]:
import os
import secrets
import requests

from maas_notebook_utils import (
    disable_tls_warnings,
    get_openshift_identity,
    load_env,
    request_start_time,
    set_kubeconfig,
    show_api_key_response,
    show_inference_response,
    show_json_response,
    show_openshift_identity,
    show_revocation_response,
)

disable_tls_warnings()

# Load local environment configuration from .env if present
env_file = load_env()
if env_file:
    print(f"Loaded environment variables from: {env_file.name}")

# Set KUBECONFIG (from environment or default ~/.kube/config)
KUBECONFIG_PATH = os.getenv("KUBECONFIG", "~/.kube/config")
set_kubeconfig(KUBECONFIG_PATH)

# Cluster & MaaS endpoint configuration
MAAS_GATEWAY_URL = os.getenv("MAAS_GATEWAY_URL", "https://maas.apps.example.com")
MAAS_API = os.getenv("MAAS_API", f"{MAAS_GATEWAY_URL}/maas-api/v1")
INFERENCE_URL = f"{MAAS_GATEWAY_URL}/v1/chat/completions"

MAAS_SUBSCRIPTION = os.getenv("MAAS_SUBSCRIPTION", "free-tier")
MODEL = os.getenv("MAAS_MODEL", "gpt-oss-20b")

### 1. User Login Flow (OpenShift CLI)
The user is authenticated with OpenShift via the `oc` CLI. The notebook retrieves the OpenShift user token using `oc whoami -t`.

In [ ]:
USERNAME, ACCESS_TOKEN, CLUSTER_SERVER = get_openshift_identity()
show_openshift_identity(USERNAME, ACCESS_TOKEN, CLUSTER_SERVER)

### 2. List Subscriptions (RHOAI MaaS API)

User lists the subscriptions they have access to by authenticating with the ***OpenShift Access Token**. Without access to one or more subscriptions, the user cannot create an API key.

To list the subscriptions, the group name configured in **MaaSSubscription** custom resources must match the user groups returned by OpenShift TokenReview.

In [ ]:
response = requests.get(
    f"{MAAS_API}/subscriptions",
    headers={"Authorization": f"Bearer {ACCESS_TOKEN}"},
    verify=False,
)

show_json_response(response)

### 3. List Models (RHOAI MaaS API call with OpenShift Access Token)

User lists the models they have access to by authenticating with the **OpenShift Access Token**.

If the user wants to access a model, the group name configured in **MaaSAuthPolicy** custom resources must match the user groups returned by OpenShift TokenReview.

In [ ]:
response = requests.get(
    f"{MAAS_API}/models",
    headers={"Authorization": f"Bearer {ACCESS_TOKEN}"},
    verify=False,
)

show_json_response(response)

### 4. API Key Creation Flow (RHOAI MaaS API)

User creates an API key by authenticating with the **OpenShift Access Token**. MaaS generates a key, stores only the hash in the database, and returns the plaintext once:

In [ ]:
KEY_NAME = f"{MAAS_SUBSCRIPTION}-{USERNAME}-e2e-{secrets.token_hex(4)}"

response = requests.post(
    f"{MAAS_API}/api-keys",
    headers={"Authorization": f"Bearer {ACCESS_TOKEN}"},
    json={
        "name": KEY_NAME,
        "description": "Temporary key for test",
        "expiresIn": "1h",
        "subscription": MAAS_SUBSCRIPTION,
    },
    verify=False,
)

MAAS_API_KEY, MAAS_API_KEY_ID = show_api_key_response(response)

### 5. Model Inference Flow

Inference requests use the API key. The external metering simulator keeps an independent **$1.00 balance per user** and charges **$0.50** after each successful inference. Run this cell repeatedly to observe the cycle: two allowed requests, a third denied request that resets the balance, and then an allowed request again.

The cell also prints only the simulator exchanges for this invocation: the entitlement GET, followed by the usage-event POST when access is allowed.

In [ ]:
started_at = request_start_time()
response = requests.post(
    INFERENCE_URL,
    headers={
        "Authorization": f"Bearer {MAAS_API_KEY}",
        "Accept": "application/json",
    },
    json={
        "model": MODEL,
        "messages": [
            {
                "role": "user",
                "content": "What is your tip for today?",
            }
        ],
    },
    verify=False,
)

show_inference_response(
    response,
    username=USERNAME,
    model=MODEL,
    started_at=started_at,
)

### 6. Revoke API Key Flow (RHOAI MaaS API)

User revokes a previously created API key.

In [ ]:
response = requests.delete(
    f"{MAAS_API}/api-keys/{MAAS_API_KEY_ID}",
    headers={
        "Authorization": f"Bearer {ACCESS_TOKEN}",
        "Accept": "application/json",
    },
    verify=False,
)

show_revocation_response(response)